In [11]:
import CoolProp.CoolProp as CP

# =============================================================================
# 1. INPUT PARAMETERS (Converted to SI Units: Pa, K, kg/s)
# =============================================================================
fluid = 'Toluene'

# Target net power output
W_dot_ORC_target = 850.0 * 1e3  # 850 kW -> Watts

# Process efficiencies
eta_OT = 0.80      # Turbine isentropic efficiency
eta_OP = 0.75      # Pump isentropic efficiency

# Component boundaries
T_cond = 50.0 + 273.15          # Condensation temperature: 50 °C -> Kelvin
T5 = 350.0 + 273.15             # Turbine inlet temperature: 350 °C -> Kelvin

# Pinch points and cooling parameters (for future heat exchanger expansion)
Delta_pp_e = 30.0
Delta_pp_cond = 20.0
Delta_pp_rec = 20.0

P_cool = 100.0 * 1e3            # 100 kPa -> Pa
T_cool_in = 20.0 + 273.15       # 20 °C -> Kelvin
T_cool_pp = 30.0 + 273.15       # 30 °C -> Kelvin
T_cool_out = T_cool_in + Delta_pp_cond + 10.0

# --- CORRECTED STANDARD LOOKUPS FOR FLUID CONSTANTS ---
# We use standard PropsSI but fetch 'TCRIT' and 'PCRIT' directly using fluid name
T_c = CP.PropsSI('TCRIT', fluid)
P_c = CP.PropsSI('PCRIT', fluid)

# Define high and low cycle pressures matching your EES equations
P_high = 0.9 * P_c              # Maximum safe pressure (90% of critical)
P_low = CP.PropsSI('P', 'T', T_cond, 'Q', 0, fluid) # Saturation pressure at T_cond

print(f"--- Cycle Boundaries Established ---")
print(f"Critical Pressure: {P_c/1e3:.2f} kPa | High Operating Pressure: {P_high/1e3:.2f} kPa")
print(f"Condensation Pressure (Low Side): {P_low/1e3:.2f} kPa")

--- Cycle Boundaries Established ---
Critical Pressure: 4126.35 kPa | High Operating Pressure: 3713.71 kPa
Condensation Pressure (Low Side): 12.29 kPa


In [12]:
# =============================================================================
# 2. STATE POINT CALCULATIONS (SI Units)
# =============================================================================

# --- State 8: Pump Inlet (Saturated Liquid at T_cond) ---
h8 = CP.PropsSI('H', 'T', T_cond, 'Q', 0, fluid)
s8 = CP.PropsSI('S', 'T', T_cond, 'Q', 0, fluid)
P8 = P_low

# --- State 9: Pump Outlet (High Pressure Subcooled Liquid) ---
# 1. Calculate ideal isentropic outlet enthalpy
ss9 = s8
hs9 = CP.PropsSI('H', 'P', P_high, 'S', ss9, fluid)
# 2. Factor in pump efficiency to get actual h9: eta_OP = (hs9 - h8) / (h9 - h8)
h9 = h8 + (hs9 - h8) / eta_OP
T9 = CP.PropsSI('T', 'P', P_high, 'H', h9, fluid)
s9 = CP.PropsSI('S', 'P', P_high, 'H', h9, fluid)

# --- State 10: Evaporator Saturated Liquid Line ---
h10 = CP.PropsSI('H', 'P', P_high, 'Q', 0, fluid)
s10 = CP.PropsSI('S', 'P', P_high, 'Q', 0, fluid)
T10 = CP.PropsSI('T', 'P', P_high, 'Q', 0, fluid)

# --- State 14: Evaporator Saturated Vapor Line ---
h14 = CP.PropsSI('H', 'P', P_high, 'Q', 1, fluid)
s14 = CP.PropsSI('S', 'P', P_high, 'Q', 1, fluid)
T14 = CP.PropsSI('T', 'P', P_high, 'Q', 1, fluid)

# --- State 5: Turbine Inlet (Superheated Vapor at T5) ---
h5 = CP.PropsSI('H', 'P', P_high, 'T', T5, fluid)
s5 = CP.PropsSI('S', 'P', P_high, 'T', T5, fluid)

# --- State 6: Turbine Outlet / Condenser Inlet ---
# 1. Calculate ideal isentropic outlet enthalpy
ss6 = s5
hs6 = CP.PropsSI('H', 'P', P_low, 'S', ss6, fluid)
# 2. Factor in turbine efficiency to get actual h6: eta_OT = (h5 - h6) / (h5 - hs6)
h6 = h5 - eta_OT * (h5 - hs6)
T6 = CP.PropsSI('T', 'P', P_low, 'H', h6, fluid)
s6 = CP.PropsSI('S', 'P', P_low, 'H', h6, fluid)

# --- State 7: Saturated Vapor Entering Condenser Phase Change ---
h7 = CP.PropsSI('H', 'T', T_cond, 'Q', 1, fluid)
s7 = CP.PropsSI('S', 'T', T_cond, 'Q', 1, fluid)
P7 = P_low

print("--- State Points Derived Accurately ---")

--- State Points Derived Accurately ---


In [13]:
# =============================================================================
# 2b. RECUPERATOR STATE CALCULATIONS (SI Units)
# =============================================================================

print("--- Integrating Recuperator Components ---")

# --- Recuperator Hot-Side Outlet (Gas traveling to Condenser) ---
# The high-pressure side liquid inlet temperature (T9) governs the lower limit
P_hot_gas_out = P_low
T_hot_gas_out = T9 + Delta_pp_rec   # T_hot_gas_out = T[9] + Delta_pp_rec

# Look up enthalpy and entropy at this defined state
h_hot_gas_out = CP.PropsSI('H', 'P', P_hot_gas_out, 'T', T_hot_gas_out, fluid)
s_hot_gas_out = CP.PropsSI('S', 'P', P_hot_gas_out, 'T', T_hot_gas_out, fluid)

# --- Recuperator Cold-Side Outlet (Preheated Liquid traveling to Evaporator) ---
P_preheated_liq_out = P_high

# Energy Balance: (h6 - h_hot_gas_out) = (h_preheated_liq_out - h9)
# Therefore: h_preheated_liq_out = h9 + (h6 - h_hot_gas_out)
h_preheated_liq_out = h9 + (h6 - h_hot_gas_out)

# Find remaining thermodynamic properties from P and our calculated enthalpy
T_preheated_liq_out = CP.PropsSI('T', 'P', P_preheated_liq_out, 'H', h_preheated_liq_out, fluid)
s_preheated_liq_out = CP.PropsSI('S', 'P', P_preheated_liq_out, 'H', h_preheated_liq_out, fluid)

print("--- Recuperator States [Hot Gas Out & Preheated Liquid Out] Calculated ---")

--- Integrating Recuperator Components ---
--- Recuperator States [Hot Gas Out & Preheated Liquid Out] Calculated ---


In [14]:
# =============================================================================
# 2c. COOLING WATER & CONDENSER STATE CALCULATIONS (SI Units)
# =============================================================================

print("--- Integrating Condenser Cooling Loop ---")

# CoolProp uses 'Water' for liquid water properties
water_fluid = 'Water'

# Look up cooling water enthalpies using the parameters defined in Section 1
h_cool_in  = CP.PropsSI('H', 'P', P_cool, 'T', T_cool_in, water_fluid)
h_cool_pp  = CP.PropsSI('H', 'P', P_cool, 'T', T_cool_pp, water_fluid)
h_cool_out = CP.PropsSI('H', 'P', P_cool, 'T', T_cool_out, water_fluid)

print("--- Cooling Water Enthalpies Evaluated Successfully ---")

--- Integrating Condenser Cooling Loop ---
--- Cooling Water Enthalpies Evaluated Successfully ---


In [15]:
# =============================================================================
# 3. COMPLETE SYSTEM BALANCES (WITH COOLING LOOP)
# =============================================================================

# Total heat rejected by the ORC working fluid in the condenser (Watts)
Q_dot_out = m_dot_orc * (h_hot_gas_out - h8)

# Condenser Energy Balance to find cooling water mass flow rate (kg/s)
# m_dot_orc * (h_hot_gas_out - h8) = m_dot_cool * (h_cool_out - h_cool_in)
m_dot_cool = Q_dot_out / (h_cool_out - h_cool_in)

# =============================================================================
# 4. FINAL COMPREHENSIVE VALIDATION PRINTOUT
# =============================================================================
print(f"==================================================")
print(f"      FINAL COMPLETE RORC SYSTEM SUMMARY         ")
print(f"==================================================")
print(f"ORC Fluid Mass Flow (m_dot_orc):   {m_dot_orc:.3f} kg/s")
print(f"Cooling Water Mass Flow (m_cool):  {m_dot_cool:.3f} kg/s")
print(f"--------------------------------------------------")
print(f"Turbine Power Output (W_dot_OT):   {W_dot_OT / 1e3:.2f} kW")
print(f"Pump Power Consumption (W_dot_OP): {W_dot_OP / 1e3:.2f} kW")
print(f"Net System Power Output:           {W_dot_ORC / 1e3:.2f} kW")
print(f"--------------------------------------------------")
print(f"Recuperator Heat Duty (Q_Rec):     {Q_dot_Rec / 1e3:.2f} kW")
print(f"Evaporator Heat Input (Q_in):      {Q_dot_orc_recup / 1e3:.2f} kW")
print(f"Condenser Heat Rejection (Q_out):  {Q_dot_out / 1e3:.2f} kW")
print(f"--------------------------------------------------")
print(f"Final Optimized Cycle Efficiency:  {Eta_orc_recup:.2f} %")
print(f"==================================================")
print(f"==================================================")

      FINAL COMPLETE RORC SYSTEM SUMMARY         
ORC Fluid Mass Flow (m_dot_orc):   4.311 kg/s
Cooling Water Mass Flow (m_cool):  14.638 kg/s
--------------------------------------------------
Turbine Power Output (W_dot_OT):   875.33 kW
Pump Power Consumption (W_dot_OP): 25.33 kW
Net System Power Output:           850.00 kW
--------------------------------------------------
Recuperator Heat Duty (Q_Rec):     961.24 kW
Evaporator Heat Input (Q_in):      2685.74 kW
Condenser Heat Rejection (Q_out):  1835.74 kW
--------------------------------------------------
Final Optimized Cycle Efficiency:  31.65 %
